In [2]:
import pandas as pd
import numpy as np
from tools import sherlock
import calendar

In [3]:
df = pd.read_excel(r"..\data\raw\taxe_de_sejour_2024.ods", engine='odf', nrows=41)

# Exploration

In [4]:
sherlock(df)

41 lignes | 10 colonnes | 0 lignes doublons
--------------------------------------------------
                                                              Type  Manquants  \
Unnamed: 0                                                   int64          0   
Date de perception de la taxe  (date du séjour      datetime64[us]          0   
A. Montant de la nuitée                                    float64          0   
B. Nombre de nuitées enregistrées                            int64          0   
C. Nombre de personnes qui ont séjournées                    int64          0   
D. Nombre d'exonérations                                     int64          0   
Motifs d'exonérations de la taxe                               str          0   
E. Prix par nuitée par personne (Hébergement no...         float64          0   
G. tarif unitaire de la taxe                               float64          0   
H. Montant de la taxe perçue ExBxG                         float64          0   

             

In [5]:
df = df.rename(str.lower, axis='columns')
df.columns = df.columns.str.strip().str.replace(' ', '_')
df = df.drop(columns=[
    'unnamed:_0', 'motifs_d\'exonérations_de_la_taxe', 'd._nombre_d\'exonérations',
    'g._tarif_unitaire_de_la_taxe', 'h._montant_de_la_taxe_perçue_exbxg', 'a._montant_de_la_nuitée'
    ])
df = df.rename(columns={
    'date_de_perception_de_la_taxe__(date_du_séjour':'date', 
    'b._nombre_de_nuitées_enregistrées':'nb_nuit',
    'c._nombre_de_personnes_qui_ont_séjournées':'nb_personnes',
    'e._prix_par_nuitée_par_personne_(hébergement_non_classé)_a/c':'prix_nuitée_par_personne'
    })

df['prix_nuitée'] = df['prix_nuitée_par_personne'] * df['nb_personnes']

In [6]:
sherlock(df)

41 lignes | 5 colonnes | 0 lignes doublons
--------------------------------------------------
                                    Type  Manquants Manquants %  Uniques
date                      datetime64[us]          0        0.0%       41
nb_nuit                            int64          0        0.0%        5
nb_personnes                       int64          0        0.0%        8
prix_nuitée_par_personne         float64          0        0.0%       40
prix_nuitée                      float64          0        0.0%       35
--------------------------------------------------
Clé potentielle sur date


In [7]:
df.head()

,date,nb_nuit,nb_personnes,prix_nuitée_par_personne,prix_nuitée
0,2024-07-16,2,3,29.080000,87.240
1,2024-07-22,5,5,21.863600,109.318
2,2024-07-27,2,6,17.914167,107.485
3,2024-07-29,1,5,24.936000,124.680
4,2024-07-30,2,9,11.246667,101.220


# Feature

In [8]:
df['mois'] = df['date'].dt.to_period('M')
df['jours_dispo'] = df.apply(lambda row: calendar.monthrange(row['date'].year, row['date'].month)[1], axis=1)
df.loc[df['mois'] == '2024-07', 'jours_dispo'] = 31 - 16 + 1

df = df.groupby(df['mois']).agg(
    occupation = ('nb_nuit', 'sum'),
    prix_moyen_nuitée = ('prix_nuitée', 'mean'),
    duree_moyenne_sejour = ('nb_nuit', 'mean'),
    nb_sejours = ('date', 'count'),
    jours_dispo_mois = ('jours_dispo', 'max'),
).reset_index()

# Taux d'occupation
df['taux_occupation'] = df['occupation'] / df['jours_dispo_mois']

# RevPAR (Revenue Per Available Room/Night)
df['RevPAR'] = df['prix_moyen_nuitée'] * df['taux_occupation']

In [15]:
# RevPAR moyen 2024 : pondéré par le nombre de nuits vendues (ADR = prix moyen par nuit vendue)
occupation_pct = df['occupation'].sum() / df['jours_dispo_mois'].sum()
adr2024 = (df['prix_moyen_nuitée'] * df['occupation']).sum() / df['occupation'].sum()
occ_moy = (df['duree_moyenne_sejour'] * df['nb_sejours']).sum() / df['nb_sejours'].sum()
revpar2024 = adr2024 * occupation_pct

print("Résultat 2024 :")
print("-"*30)
print(f"Occupation % = {occupation_pct*100:.2f} %")
print(f"ADR 2024 = {adr2024:.2f} €")
print(f"RevPAR 2024 = {revpar2024:.2f} €")
print(f"Durée moyenne de séjour = {occ_moy:.2f} jours")

Résultat 2024 :
------------------------------
Occupation % = 50.30 %
ADR 2024 = 98.28 €
RevPAR 2024 = 49.43 €
Durée moyenne de séjour = 2.07 jours


In [10]:
df

,mois,occupation,prix_moyen_nuitée,duree_moyenne_sejour,nb_sejours,jours_dispo_mois,taux_occupation,RevPAR
0,2024-07,12,105.988600,2.400000,5,16,0.750000,79.491450
1,2024-08,31,103.763646,1.937500,16,31,1.000000,103.763646
2,2024-09,8,105.437500,2.000000,4,30,0.266667,28.116667
3,2024-10,16,106.550000,2.000000,8,31,0.516129,54.993548
4,2024-11,11,77.988571,1.571429,7,30,0.366667,28.595810
5,2024-12,7,65.565714,7.000000,1,31,0.225806,14.805161


### Export

In [11]:
df.to_csv(r"..\data\processed\tx_occupation.csv", index=False, sep=';', decimal=',', encoding='utf-8-sig')